# Stage 2 — Local/Centralized Training (Optimized)


## Cell 40 — Imports


In [ ]:
# Cell 40: Stage 2 imports

import os
import json
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

ImageFile.LOAD_TRUNCATED_IMAGES = True


## Cell 41 — Configuration


In [ ]:
# Cell 41: Stage 2 configuration

SEED = 42

BASE_OUTPUT = Path(r"D:\RMS\processed_idrid")

METADATA_DIR = BASE_OUTPUT / "metadata"
MODEL_DIR = BASE_OUTPUT / "models"
RESULTS_DIR = BASE_OUTPUT / "results"
PLOTS_DIR = BASE_OUTPUT / "training_plots"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Binary classification (DR vs No-DR) based on base paper
NUM_CLASSES = 2

# Increased resolution to 512x512 to preserve DR features like microaneurysms
IMAGE_SIZE = 512

BATCH_SIZE = 8       # Reduced batch size slightly due to larger image size
NUM_WORKERS = 0      # safest for Windows/Jupyter

EPOCHS = 40
WARMUP_EPOCHS = 3    # Freeze backbone for first 3 epochs

# Differentiated Learning Rates
LEARNING_RATE = 1e-3   # Higher LR for randomly initialized classification head
BACKBONE_LR = 1e-5     # Very low LR to gently fine-tune the pretrained backbone
WEIGHT_DECAY = 1e-3

PATIENCE = 10

# Target based on base paper's ResNet50 accuracy without noise
TARGET_ACCURACY = 0.8305


## Cell 42 — Reproducibility + device


In [ ]:
# Cell 42: Reproducibility and device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Cell 43 — Load Stage 1 metadata


In [ ]:
# Cell 43: Load train/validation/test metadata

TRAIN_META = METADATA_DIR / "train.csv"
VAL_META = METADATA_DIR / "val.csv"
TEST_META = METADATA_DIR / "test.csv"

for file_path in [TRAIN_META, VAL_META, TEST_META]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required Stage 1 file not found:\n{file_path}\n\n"
            "Run Stage 1 successfully before starting Stage 2."
        )

train_df = pd.read_csv(TRAIN_META)
val_df = pd.read_csv(VAL_META)
test_df = pd.read_csv(TEST_META)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Official Test:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())


## Cell 44 — Verify metadata


In [ ]:
# Cell 44: Verify metadata

REQUIRED_COLUMNS = [
    "Image name",
    "Retinopathy grade",
    "image_path"
]

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{name} metadata missing columns: {missing}")

    missing_paths = df["image_path"].isna().sum()
    if missing_paths > 0:
        raise ValueError(f"{name} contains {missing_paths} missing image paths.")

    missing_files = [p for p in df["image_path"] if not Path(p).exists()]
    if missing_files:
        raise FileNotFoundError(f"{name}: {len(missing_files)} image files do not exist.")


## Cell 45 — Check class distributions (Binary)


In [ ]:
# Cell 45: Class distribution

print("===== TRAIN =====")
print((train_df["Retinopathy grade"] > 0).astype(int).value_counts().sort_index())

print("\n===== VALIDATION =====")
print((val_df["Retinopathy grade"] > 0).astype(int).value_counts().sort_index())

print("\n===== OFFICIAL TEST =====")
print((test_df["Retinopathy grade"] > 0).astype(int).value_counts().sort_index())


# Data preprocessing
## Cell 46 — Normalization


In [ ]:
# Cell 46: Image normalization

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


## Cell 47 — Training and evaluation transforms (Enhanced)


In [ ]:
# Cell 47: Transforms

def get_train_transform(image_size=512):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=45),
        transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2), shear=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

def get_eval_transform(image_size=512):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform = get_eval_transform(IMAGE_SIZE)


# Dataset
## Cell 48 — Dataset class (Binary labels)


In [ ]:
# Cell 48: IDRiD Dataset

class IDRiDLocalDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_path = row["image_path"]

        # Base paper: binary classification: DR vs No-DR
        grade = int(row["Retinopathy grade"])
        label = 1 if grade > 0 else 0

        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return (image, label, row["Image name"])


## Cell 49 — Create datasets


In [ ]:
# Cell 49: Create datasets

train_dataset = IDRiDLocalDataset(train_df, transform=train_transform)
val_dataset = IDRiDLocalDataset(val_df, transform=eval_transform)
test_dataset = IDRiDLocalDataset(test_df, transform=eval_transform)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


## Cell 50 — DataLoaders


In [ ]:
# Cell 50: DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


## Cell 51 — Verify batch


In [ ]:
# Cell 51: Verify batch

images, labels, names = next(iter(train_loader))

print("Image shape :", images.shape)
print("Labels shape:", labels.shape)
print("First names:")
print(list(names[:5]))

assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)


# Class weights
## Cell 52 — Calculate binary weights


In [ ]:
# Cell 52: Class weights

train_labels = (train_df["Retinopathy grade"].astype(int) > 0).astype(int).values

class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
total_samples = len(train_labels)

class_weights = (
    total_samples /
    (NUM_CLASSES * np.maximum(class_counts, 1))
)

class_weights = torch.tensor(
    class_weights, dtype=torch.float32
).to(DEVICE)

print("Class counts:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count}")

print("\nClass weights:")
for i, weight in enumerate(class_weights):
    print(f"Class {i}: {weight.item():.4f}")


# Loss
## Cell 53 — Focal Loss function


In [ ]:
# Cell 53: Focal Loss Function

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss(alpha=class_weights, gamma=2.0)


# Metrics
## Cell 54 — Metric function


In [ ]:
# Cell 54: Metric calculation

def calculate_metrics(labels, predictions, probabilities):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    results = {}

    results["accuracy"] = accuracy_score(labels, predictions)
    results["balanced_accuracy"] = balanced_accuracy_score(labels, predictions)
    results["precision"] = precision_score(labels, predictions, average="macro", zero_division=0)
    results["recall"] = recall_score(labels, predictions, average="macro", zero_division=0)
    results["macro_f1"] = f1_score(labels, predictions, average="macro", zero_division=0)
    results["weighted_f1"] = f1_score(labels, predictions, average="weighted", zero_division=0)

    try:
        results["auc"] = roc_auc_score(labels, probabilities[:, 1])
    except ValueError:
        results["auc"] = np.nan

    return results


# Evaluation
## Cell 55 — Evaluation function


In [ ]:
# Cell 55: Model evaluation

def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(probabilities, dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    probabilities = np.concatenate(all_probabilities, axis=0)
    metrics = calculate_metrics(all_labels, all_predictions, probabilities)
    metrics["loss"] = total_loss / total_samples
    metrics["labels"] = np.array(all_labels)
    metrics["predictions"] = np.array(all_predictions)
    metrics["probabilities"] = probabilities
    return metrics


# Models (Matching base paper: AlexNet, ResNet50, SqueezeNet1.1, VGG16)
## Cell 56 — Model Definitions


In [ ]:
# Cell 56: Model Definitions

def create_alexnet():
    model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.50),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model

def create_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.50),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model

def create_squeezenet1_1():
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)
    model.classifier[1] = nn.Conv2d(512, NUM_CLASSES, kernel_size=(1,1), stride=(1,1))
    model.num_classes = NUM_CLASSES
    return model

def create_vgg16():
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.50),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model

models_dict = {
    "AlexNet": create_alexnet().to(DEVICE),
    "ResNet50": create_resnet50().to(DEVICE),
    "SqueezeNet1.1": create_squeezenet1_1().to(DEVICE),
    "VGG16": create_vgg16().to(DEVICE)
}


# Training function
## Cell 57 — Training function (Warmup + Differential LR)


In [ ]:
# Cell 57: Local training function

def train_local_model(model, model_name, train_loader, val_loader, criterion, epochs, lr, backbone_lr, weight_decay, patience, warmup_epochs):
    
    # Identify backbone and classifier for differential learning rates
    if hasattr(model, 'fc'):
        classifier_params = list(model.fc.parameters())
        backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    elif hasattr(model, 'classifier'):
        classifier_params = list(model.classifier.parameters())
        backbone_params = [p for n, p in model.named_parameters() if not n.startswith('classifier.')]
    else:
        classifier_params = list(model.parameters())
        backbone_params = []

    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr},
        {'params': classifier_params, 'lr': lr}
    ], weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    history = {
        "train_loss": [], "train_accuracy": [],
        "val_loss": [], "val_accuracy": [],
        "val_balanced_accuracy": [], "val_macro_f1": [], "val_auc": []
    }

    best_state = copy.deepcopy(model.state_dict())
    best_accuracy = -1.0
    best_f1 = -1.0
    best_epoch = 0
    patience_counter = 0

    model_path = MODEL_DIR / f"{model_name}_best.pt"
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        
        # Progressive Unfreezing (Warmup Phase)
        if epoch <= warmup_epochs:
            for param in backbone_params:
                param.requires_grad = False
            if epoch == 1:
                print(f"Warmup Phase: Backbone frozen for {warmup_epochs} epochs.")
        elif epoch == warmup_epochs + 1:
            for param in backbone_params:
                param.requires_grad = True
            print("Fine-tuning Phase: Backbone unfrozen.")
            
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels, _ in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_loss = running_loss / total
        train_accuracy = correct / total

        val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_balanced_accuracy"].append(val_metrics["balanced_accuracy"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["val_auc"].append(val_metrics["auc"])

        cls_lr = optimizer.param_groups[1]["lr"] if len(optimizer.param_groups) > 1 else optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Acc: {train_accuracy*100:.2f}% | "
            f"Val Acc: {val_metrics['accuracy']*100:.2f}% | "
            f"Val F1: {val_metrics['macro_f1']*100:.2f}% | "
            f"Val AUC: {val_metrics['auc']:.4f} | "
            f"Cls LR: {cls_lr:.2e}"
        )

        improved = (
            val_metrics["accuracy"] > best_accuracy or 
            (abs(val_metrics["accuracy"] - best_accuracy) < 1e-8 and val_metrics["macro_f1"] > best_f1)
        )

        if improved:
            best_accuracy = val_metrics["accuracy"]
            best_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, model_path)
            patience_counter = 0
            print(f"   Best model saved | Val Accuracy = {best_accuracy*100:.2f}%")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("\nEarly stopping")
            break

    elapsed = time.time() - start_time
    model.load_state_dict(best_state)

    print("\n====================================")
    print(model_name)
    print("====================================")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation accuracy: {best_accuracy*100:.2f}%")
    print(f"Training time: {elapsed/60:.2f} minutes")

    return model, history, best_accuracy, best_epoch


# Train Models
## Cell 58


In [ ]:
# Cell 58: Train all models

results_dict = {}

for name, model in models_dict.items():
    print("\n" + "=" * 50)
    print(f"Training {name}")
    print("=" * 50)
    
    trained_model, history, best_acc, best_epoch = train_local_model(
        model=model,
        model_name=f"{name.lower()}_idrid",
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        backbone_lr=BACKBONE_LR,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        warmup_epochs=WARMUP_EPOCHS
    )
    
    results_dict[name] = {
        "model": trained_model,
        "history": history,
        "best_acc": best_acc,
        "best_epoch": best_epoch
    }


# Validation comparison
## Cell 59


In [ ]:
# Cell 59: Compare models on validation set

comparison_data = []

for name, res in results_dict.items():
    history = res["history"]
    comparison_data.append({
        "Model": name,
        "Best Val Accuracy": res["best_acc"],
        "Best Val Macro F1": max(history["val_macro_f1"]),
        "Best Val Balanced Accuracy": max(history["val_balanced_accuracy"]),
        "Best Val AUC": np.nanmax(history["val_auc"]),
        "Best Epoch": res["best_epoch"]
    })

comparison = pd.DataFrame(comparison_data)

display(
    comparison.style.format({
        "Best Val Accuracy": "{:.2%}",
        "Best Val Macro F1": "{:.2%}",
        "Best Val Balanced Accuracy": "{:.2%}",
        "Best Val AUC": "{:.4f}"
    })
)


# Detailed validation evaluation
## Cell 60 — Function


In [ ]:
# Cell 60: Detailed evaluation report

def print_detailed_report(model, loader, model_name):
    metrics = evaluate_model(model, loader, criterion, DEVICE)
    labels = metrics["labels"]
    predictions = metrics["predictions"]

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)
    print(f"Accuracy          : {metrics['accuracy']*100:.2f}%")
    print(f"Balanced Accuracy : {metrics['balanced_accuracy']*100:.2f}%")
    print(f"Macro Precision   : {metrics['precision']*100:.2f}%")
    print(f"Macro Recall      : {metrics['recall']*100:.2f}%")
    print(f"Macro F1          : {metrics['macro_f1']*100:.2f}%")
    print(f"Weighted F1       : {metrics['weighted_f1']*100:.2f}%")
    print(f"Macro AUC         : {metrics['auc']:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            labels,
            predictions,
            labels=[0, 1],
            target_names=["No DR", "DR"],
            zero_division=0
        )
    )

    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["No DR", "DR"],
        yticklabels=["No DR", "DR"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return metrics


# Select best model
## Cell 61


In [ ]:
# Cell 61: Select best model using validation accuracy

best_model_name = comparison.loc[comparison["Best Val Accuracy"].idxmax()]["Model"]
BEST_MODEL_NAME = best_model_name
BEST_MODEL = results_dict[best_model_name]["model"]
BEST_MODEL_PATH = MODEL_DIR / f"{best_model_name.lower()}_idrid_best.pt"

print("Selected model:", BEST_MODEL_NAME)
print("Saved model:", BEST_MODEL_PATH)


# IMPORTANT: Official test evaluation

This is the **only place** where we evaluate the official 103 images for this Stage 2 experiment.

## Cell 62


In [ ]:
# Cell 62: FINAL official IDRiD test evaluation

print("=" * 70)
print("FINAL OFFICIAL TEST EVALUATION")
print("=" * 70)

test_metrics = print_detailed_report(
    BEST_MODEL,
    test_loader,
    f"{BEST_MODEL_NAME} - Official IDRiD Test"
)


# Save final local results
## Cell 63


In [ ]:
# Cell 63: Save Stage 2 results

final_results = {
    "best_model": BEST_MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "best_validation_accuracy": float(results_dict[best_model_name]["best_acc"]),
    "official_test_accuracy": float(test_metrics["accuracy"]),
    "official_test_balanced_accuracy": float(test_metrics["balanced_accuracy"]),
    "official_test_macro_precision": float(test_metrics["precision"]),
    "official_test_macro_recall": float(test_metrics["recall"]),
    "official_test_macro_f1": float(test_metrics["macro_f1"]),
    "official_test_weighted_f1": float(test_metrics["weighted_f1"]),
    "official_test_auc": None if np.isnan(test_metrics["auc"]) else float(test_metrics["auc"])
}

with open(RESULTS_DIR / "stage2_local_results.json", "w") as f:
    json.dump(final_results, f, indent=4)

print(json.dumps(final_results, indent=4))


# Cell 64 — Target check


In [ ]:
# Cell 64: Check local target

test_accuracy = test_metrics["accuracy"]

print("=" * 70)
print("STAGE 2 LOCAL TARGET")
print("=" * 70)
print(f"Official Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Target Accuracy       : {TARGET_ACCURACY * 100:.2f}%")

if test_accuracy >= TARGET_ACCURACY:
    print("\nTarget achieved! Official test accuracy is above base paper baseline.")
else:
    print("\nTarget missed! Official test accuracy is below base paper baseline.")
    print("\nDo NOT tune directly on the official test set.")
    print("Use the validation set for further optimization.")


# Next Steps

**Do not move directly to FedAvg if accuracy is below target.**

For your target, the next local optimization should be done on the **validation set**.

Because your IDRiD set has only **413 official training images**, a reported strong result must also be checked for stability; otherwise a single train/validation split can make the result look stronger than it really is.

Once you have the validation results that meet the base paper baseline of 83.05%, we can decide the exact local model and resolution for **Stage 3: FedAvg**.
